# COMP6008 Smart Irrigation RL — Group Notebook

**Location:** Kyabram, Victoria  
**Algorithms:** DDPG, TD3 and SAC  
**Status:** Shared environment implemented; agent sections are templates.

This is the group's shared working notebook. Keep the environment, data splits and evaluation rules identical for all agents. Each member should work only in their assigned agent section until integration.

## 0. Group allocation

Fill this table in after the group confirms responsibilities.

| Component | Group member | Branch or notebook copy | Status |
|---|---|---|---|
| Shared environment and integration | TODO | TODO | Environment ready |
| DDPG | TODO | TODO | Not started |
| TD3 | TODO | TODO | Not started |
| SAC | TODO | TODO | Not started |

> The environment cells are shared and should not be independently changed inside an agent branch. Proposed environment changes must be agreed by the group first.

## 1. Setup

Run this installation cell once per Colab session.

In [ ]:
%pip install -q "gymnasium>=1.0,<2" "numpy>=1.22" "torch>=2.0" "pandas>=2.0" "matplotlib>=3.7"

In [ ]:
from __future__ import annotations

import json
import random
from collections import deque
from dataclasses import dataclass
from datetime import date, timedelta
from pathlib import Path
from typing import Any
from zipfile import ZipFile

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from gymnasium import spaces
from gymnasium.utils.env_checker import check_env

print('Gymnasium:', gym.__version__)
print('PyTorch:', torch.__version__)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## 2. Data location

The environment needs `Irrigation_Data_Audit.zip`, already stored in the GitHub repository under `data/`. If Colab does not load the repository files automatically, upload that ZIP using the Colab Files panel. Do not upload a different climate version without recording the change.

In [ ]:
DATA_CANDIDATES = [
    Path('data/Irrigation_Data_Audit.zip'),
    Path('Irrigation_Data_Audit.zip'),
    Path('/content/Irrigation_Data_Audit.zip'),
    Path('irrigation_audit/kyabram_080091_daily.json'),
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        'Upload Irrigation_Data_Audit.zip, then rerun this cell. '
        'The expected Colab path is /content/Irrigation_Data_Audit.zip.'
    )

print('Using:', DATA_PATH.resolve())

## 3. Fixed shared settings and agent placeholders

The `None` values are intentional. Fill selected agent settings only after the declared validation-only tuning runs. Do not use test-season results to choose them.

In [ ]:
TRAIN_YEARS = tuple(range(2005, 2017))
VALIDATION_YEARS = tuple(range(2017, 2021))
TEST_YEARS = tuple(range(2021, 2025))
SEASON_SPLITS = {
    'train': TRAIN_YEARS,
    'validation': VALIDATION_YEARS,
    'test': TEST_YEARS,
    'all': TRAIN_YEARS + VALIDATION_YEARS + TEST_YEARS,
}

@dataclass(frozen=True)
class EnvironmentConfig:
    capacity_mm: float = 111.33333333333334
    initial_storage_fraction: float = 0.80
    max_irrigation_mm: float = 20.0
    irrigation_efficiency: float = 0.85
    season_length_days: int = 150
    planting_month: int = 11
    planting_day: int = 1
    planting_day_offset: int = 0
    kc_multiplier: float = 1.0
    irrigation_weight: float = 1.0
    shortage_weight: float = 10.0
    drainage_weight: float = 1.0
    rainfall_scale_mm: float = 20.0
    eto_scale_mm: float = 10.0
    history_days: int = 7

ENV_CONFIG = EnvironmentConfig()

SHARED_TRAINING = {
    'gamma': 0.995,
    'tau': 0.005,
    'replay_capacity': 100_000,
    'batch_size': 128,
    'random_exploration_steps': 1_500,
    'hidden_sizes': (128, 128),
    'final_training_steps': 100_000,
    'tuning_steps': 30_000,
    'evaluation_interval': 5_000,
    'final_seeds': (11, 22, 33, 44, 55),
    'tuning_seeds': (101, 202),
}

AGENT_SETTINGS = {
    'ddpg': {
        'owner': None,
        'learning_rate_options': (1e-4, 3e-4),
        'exploration_std_options': (0.1, 0.2),
        'selected_learning_rate': None,
        'selected_exploration_std': None,
    },
    'td3': {
        'owner': None,
        'learning_rate_options': (1e-4, 3e-4),
        'exploration_std_options': (0.1, 0.2),
        'selected_learning_rate': None,
        'selected_exploration_std': None,
        'policy_delay': 2,
        'target_smoothing_std': 0.2,
        'target_noise_clip': 0.5,
    },
    'sac': {
        'owner': None,
        'learning_rate_options': (1e-4, 3e-4),
        'alpha_options': (0.1, 0.2),
        'selected_learning_rate': None,
        'selected_alpha': None,
        'log_std_bounds': None,  # document before SAC training
    },
}

AGENT_SETTINGS

## 4. Shared Gymnasium environment

The action is selected before the current day's rainfall and ETo are revealed. The next observation contains only completed weather history.

In [ ]:
KYABRAM_MEMBER = 'irrigation_audit/kyabram_080091_daily.json'

class KyabramIrrigationEnv(gym.Env):
    metadata = {'render_modes': []}

    def __init__(self, climate_path=DATA_PATH, split='train', config=ENV_CONFIG):
        super().__init__()
        if split not in SEASON_SPLITS:
            raise ValueError(f'Unknown split: {split}')
        self.config = config
        self.split = split
        self.allowed_years = SEASON_SPLITS[split]
        records = self._load_records(Path(climate_path))
        self._weather, self._dates = self._build_seasons(records)
        self.action_space = spaces.Box(-1.0, 1.0, shape=(1,), dtype=np.float32)
        self.observation_space = spaces.Box(
            low=np.zeros(18, dtype=np.float32),
            high=np.array([1.0, 1.0, 1.1, 1.0] + [np.inf] * 14, dtype=np.float32),
            dtype=np.float32,
        )
        self.season_year = None
        self.day = config.season_length_days
        self.storage_mm = 0.0
        self.previous_irrigation_mm = 0.0
        self._terminated = True

    @staticmethod
    def _load_records(path):
        if path.is_dir():
            path = path / 'kyabram_080091_daily.json'
        if path.suffix.lower() == '.zip':
            with ZipFile(path) as archive:
                payload = json.loads(archive.read(KYABRAM_MEMBER))
        else:
            payload = json.loads(path.read_text())
        records = {}
        for index, row in enumerate(payload):
            try:
                current_date = date.fromisoformat(row['date'])
                rain = float(row['rain_mm'])
                eto = float(row['eto_mm'])
            except (KeyError, TypeError, ValueError) as exc:
                raise ValueError(f'Malformed climate row {index}') from exc
            if current_date in records or not np.isfinite([rain, eto]).all():
                raise ValueError(f'Invalid climate input on {current_date}')
            if rain < 0 or eto < 0:
                raise ValueError(f'Negative climate input on {current_date}')
            records[current_date] = (rain, eto)
        return records

    def _build_seasons(self, records):
        weather, dates = {}, {}
        block_length = self.config.history_days + self.config.season_length_days
        for year in SEASON_SPLITS['all']:
            planting = date(year, self.config.planting_month, self.config.planting_day)
            planting += timedelta(days=self.config.planting_day_offset)
            start = planting - timedelta(days=self.config.history_days)
            block_dates = tuple(start + timedelta(days=i) for i in range(block_length))
            missing = [day for day in block_dates if day not in records]
            if missing:
                raise ValueError(f'Season {year} is missing {missing[0]}')
            weather[year] = np.asarray([records[day] for day in block_dates], dtype=np.float64)
            dates[year] = block_dates
        return weather, dates

    def _kc(self, day):
        if day >= 150:
            return 0.0
        if day < 30:
            base = 0.30
        elif day < 70:
            base = 0.30 + 0.90 * (day - 29) / 40
        elif day < 120:
            base = 1.20
        else:
            base = 1.20 - 0.85 * (day - 119) / 30
        return self.config.kc_multiplier * base

    def _observation(self):
        history = self._weather[self.season_year][self.day:self.day + 7][::-1]
        observation = np.concatenate([
            np.array([
                self.storage_mm / self.config.capacity_mm,
                self.day / 150,
                self._kc(self.day) / 1.2,
                self.previous_irrigation_mm / 20,
            ]),
            history[:, 0] / self.config.rainfall_scale_mm,
            history[:, 1] / self.config.eto_scale_mm,
        ]).astype(np.float32)
        if not np.isfinite(observation).all() or not self.observation_space.contains(observation):
            raise RuntimeError('Invalid observation produced')
        return observation

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        options = {} if options is None else dict(options)
        requested_year = options.get('season_year')
        if requested_year is None:
            self.season_year = self.allowed_years[int(self.np_random.integers(len(self.allowed_years)))]
        elif int(requested_year) in self.allowed_years:
            self.season_year = int(requested_year)
        else:
            raise ValueError(f'Season {requested_year} is not in the {self.split} split')
        self.day = 0
        self.storage_mm = self.config.initial_storage_fraction * self.config.capacity_mm
        self.previous_irrigation_mm = 0.0
        self._terminated = False
        decision_date = self._dates[self.season_year][7]
        info = {'split': self.split, 'season_year': self.season_year, 'decision_date': decision_date.isoformat()}
        return self._observation(), info

    def step(self, action):
        if self._terminated:
            raise RuntimeError('Episode ended; call reset()')
        action = np.asarray(action, dtype=np.float32).reshape(-1)
        if action.shape != (1,) or not self.action_space.contains(action):
            raise ValueError('Expected one finite action in [-1, 1]')

        irrigation = 10.0 * (float(action[0]) + 1.0)
        weather_index = 7 + self.day
        rain, eto = self._weather[self.season_year][weather_index]
        decision_date = self._dates[self.season_year][weather_index]
        kc = self._kc(self.day)
        potential_et = kc * float(eto)
        depletion = float(np.clip(0.55 + 0.04 * (5.0 - potential_et), 0.1, 0.8))
        minimum = (1.0 - depletion) * self.config.capacity_mm

        storage_before = self.storage_mm
        incoming = storage_before + float(rain) + self.config.irrigation_efficiency * irrigation
        drainage = max(0.0, incoming - self.config.capacity_mm)
        wet_storage = incoming - drainage
        stress_coefficient = min(1.0, wet_storage / minimum)
        actual_et = min(wet_storage, stress_coefficient * potential_et)
        storage_after = wet_storage - actual_et
        shortage = max(0.0, minimum - storage_after)

        reward = -(
            self.config.irrigation_weight * irrigation / 20.0
            + self.config.shortage_weight * shortage / minimum
            + self.config.drainage_weight * drainage / 20.0
        )
        self.storage_mm = storage_after
        self.previous_irrigation_mm = irrigation
        self.day += 1
        self._terminated = self.day == 150
        residual = storage_after - (storage_before + rain + self.config.irrigation_efficiency * irrigation - drainage - actual_et)
        info = {
            'season_year': self.season_year, 'completed_day': self.day,
            'decision_date': decision_date.isoformat(), 'irrigation_mm': irrigation,
            'rain_mm': float(rain), 'eto_mm': float(eto), 'kc': kc,
            'potential_et_mm': potential_et, 'actual_et_mm': actual_et,
            'minimum_storage_mm': minimum, 'storage_before_mm': storage_before,
            'storage_after_mm': storage_after, 'shortage_mm': shortage,
            'drainage_mm': drainage, 'balance_residual_mm': residual,
        }
        return self._observation(), float(reward), self._terminated, False, info

## 5. Validate the shared environment

Run this before anyone begins agent training. The Gymnasium warning about infinite weather upper bounds is expected because rainfall and ETo are checked for finiteness but not clipped using test-set maxima.

In [ ]:
env = KyabramIrrigationEnv(split='train')
check_env(env, skip_render_check=True)

observation, reset_info = env.reset(seed=11, options={'season_year': 2005})
assert observation.shape == (18,)
assert observation.dtype == np.float32
assert env.observation_space.contains(observation)

rng = np.random.default_rng(11)
max_balance_error = 0.0
episode_return = 0.0
for step in range(150):
    action = rng.uniform(-1, 1, size=1).astype(np.float32)
    observation, reward, terminated, truncated, info = env.step(action)
    assert env.observation_space.contains(observation)
    assert np.isfinite(reward) and reward <= 0
    assert not truncated
    max_balance_error = max(max_balance_error, abs(info['balance_residual_mm']))
    episode_return += reward

assert terminated and step == 149
assert max_balance_error < 1e-10
print({'reset': reset_info, 'episode_return': episode_return, 'max_balance_error_mm': max_balance_error})

## 6. Shared utilities

All agents should use the same seed helper, replay record and network size. Agent owners may add algorithm-specific networks inside their own sections.

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class ReplayBuffer:
    def __init__(self, capacity=SHARED_TRAINING['replay_capacity']):
        self.memory = deque(maxlen=capacity)

    def add(self, observation, action, reward, next_observation, terminated):
        self.memory.append((observation, action, reward, next_observation, terminated))

    def sample(self, batch_size=SHARED_TRAINING['batch_size']):
        batch = random.sample(self.memory, batch_size)
        observations, actions, rewards, next_observations, terminals = map(np.asarray, zip(*batch))
        return tuple(torch.as_tensor(value, dtype=torch.float32, device=DEVICE) for value in (
            observations, actions, rewards[:, None], next_observations, terminals[:, None]
        ))

    def __len__(self):
        return len(self.memory)

def build_mlp(input_size, output_size, output_activation=None):
    h1, h2 = SHARED_TRAINING['hidden_sizes']
    layers = [nn.Linear(input_size, h1), nn.ReLU(), nn.Linear(h1, h2), nn.ReLU(), nn.Linear(h2, output_size)]
    if output_activation is not None:
        layers.append(output_activation())
    return nn.Sequential(*layers)

def soft_update(target, source, tau=SHARED_TRAINING['tau']):
    with torch.no_grad():
        for target_parameter, source_parameter in zip(target.parameters(), source.parameters()):
            target_parameter.mul_(1.0 - tau).add_(source_parameter, alpha=tau)

## 7. Benchmark policies

Use these to check whether learned policies improve on obvious controls.

In [ ]:
def zero_irrigation_policy(observation):
    return np.array([-1.0], dtype=np.float32)

def maximum_irrigation_policy(observation):
    return np.array([1.0], dtype=np.float32)

def threshold_policy(observation, capacity=ENV_CONFIG.capacity_mm):
    storage = float(observation[0]) * capacity
    if storage < 0.65 * capacity:
        irrigation = min(20.0, (0.90 * capacity - storage) / 0.85)
    else:
        irrigation = 0.0
    return np.array([irrigation / 10.0 - 1.0], dtype=np.float32)

def evaluate_policy(policy, split='validation', gamma=SHARED_TRAINING['gamma']):
    evaluation_env = KyabramIrrigationEnv(split=split)
    rows = []
    for year in SEASON_SPLITS[split]:
        observation, _ = evaluation_env.reset(options={'season_year': year})
        discounted_return = 0.0
        undiscounted_return = 0.0
        irrigation_total = shortage_total = drainage_total = 0.0
        for day in range(150):
            action = policy(observation)
            observation, reward, terminated, truncated, info = evaluation_env.step(action)
            discounted_return += (gamma ** day) * reward
            undiscounted_return += reward
            irrigation_total += info['irrigation_mm']
            shortage_total += info['shortage_mm']
            drainage_total += info['drainage_mm']
        rows.append({
            'split': split, 'season_year': year,
            'discounted_return': discounted_return,
            'undiscounted_return': undiscounted_return,
            'irrigation_mm': irrigation_total, 'shortage_mm_days': shortage_total,
            'drainage_mm': drainage_total,
        })
    return pd.DataFrame(rows)

benchmark_results = pd.concat([
    evaluate_policy(zero_irrigation_policy).assign(policy='zero'),
    evaluate_policy(maximum_irrigation_policy).assign(policy='maximum'),
    evaluate_policy(threshold_policy).assign(policy='threshold'),
])
benchmark_results.groupby('policy').mean(numeric_only=True)

## 8. DDPG — member work area

**Owner:** TODO  
**Required outputs:** actor, critic, target networks, bounded Gaussian exploration, update method, deterministic evaluation action and saved tuning results.

Reuse `ReplayBuffer`, `build_mlp`, `soft_update`, `SHARED_TRAINING` and the shared environment. Do not redefine the simulator.

In [ ]:
# DDPG IMPLEMENTATION PLACEHOLDER
# Selected only after validation tuning:
DDPG_LEARNING_RATE = AGENT_SETTINGS['ddpg']['selected_learning_rate']
DDPG_EXPLORATION_STD = AGENT_SETTINGS['ddpg']['selected_exploration_std']

class DDPGAgent:
    # TODO: implement using the Practical 9 actor-critic structure and off-policy replay ideas.
    pass

## 9. TD3 — member work area

**Owner:** TODO  
**Required outputs:** actor, two critics, target networks, bounded exploration, target-policy smoothing, delayed actor updates, deterministic evaluation action and saved tuning results.

In [ ]:
# TD3 IMPLEMENTATION PLACEHOLDER
TD3_LEARNING_RATE = AGENT_SETTINGS['td3']['selected_learning_rate']
TD3_EXPLORATION_STD = AGENT_SETTINGS['td3']['selected_exploration_std']
TD3_POLICY_DELAY = AGENT_SETTINGS['td3']['policy_delay']
TD3_TARGET_SMOOTHING_STD = AGENT_SETTINGS['td3']['target_smoothing_std']
TD3_TARGET_NOISE_CLIP = AGENT_SETTINGS['td3']['target_noise_clip']

class TD3Agent:
    # TODO: implement twin critics, minimum target and delayed policy updates.
    pass

## 10. SAC — member work area

**Owner:** TODO  
**Required outputs:** tanh-squashed Gaussian policy, log-probability correction, two Q-functions, separate value network and target, fixed entropy coefficient, deterministic evaluation action and saved tuning results.

In [ ]:
# SAC IMPLEMENTATION PLACEHOLDER
SAC_LEARNING_RATE = AGENT_SETTINGS['sac']['selected_learning_rate']
SAC_ALPHA = AGENT_SETTINGS['sac']['selected_alpha']
SAC_LOG_STD_BOUNDS = AGENT_SETTINGS['sac']['log_std_bounds']

class SACAgent:
    # TODO: implement the agreed 2018 SAC formulation with a separate value network.
    pass

## 11. Shared training and evaluation contract

Each completed agent should provide: `select_action(observation, evaluate=False)`, `update(replay_buffer)` and `save(path)`. Training code must distinguish a true season termination from an external interaction-budget cut-off.

In [ ]:
def train_agent(agent, seed, total_steps, agent_name):
    """Shared loop to complete after the three agent interfaces are agreed."""
    # TODO: add collection, warm-up, one update per step, validation checkpoints,
    # timing, checkpoint saving and the shared result schema.
    raise NotImplementedError('Complete after agent interfaces are merged')

RESULT_COLUMNS = [
    'agent', 'configuration', 'seed', 'split', 'season_year', 'training_steps',
    'discounted_return', 'undiscounted_return', 'irrigation_mm',
    'days_below_target', 'shortage_mm_days', 'actual_to_potential_et_ratio',
    'drainage_mm', 'runtime_seconds', 'status',
]
results = pd.DataFrame(columns=RESULT_COLUMNS)
results

## 12. Tuning record

Use validation seasons only. Select the highest mean discounted environmental return at 30,000 interactions across tuning seeds 101 and 202 and all four validation seasons. Break exact ties using lower irrigation, then the predeclared configuration order.

In [ ]:
tuning_record = pd.DataFrame(columns=[
    'agent', 'configuration', 'seed', 'validation_mean_discounted_return',
    'validation_mean_irrigation_mm', 'numerically_valid', 'notes',
])
tuning_record

## 13. Final comparison and plots

Add learning curves, seed-level summaries, test-season tables and robustness results here only after settings are frozen. Report all five seeds, not only the best run.

In [ ]:
# FINAL RESULTS AND PLOTS PLACEHOLDER
# TODO: load saved result files rather than relying only on notebook memory.

## 14. Reproducibility checklist

Before the final experiments, confirm:

- [ ] All three agents use this unchanged environment and the same result schema.
- [ ] The Gymnasium validation cell passes.
- [ ] Agent-specific `None` values are filled from validation tuning only.
- [ ] Test seasons have not been used for tuning or checkpoint selection.
- [ ] Seeds and training budgets match the fixed settings.
- [ ] Terminal transitions do not bootstrap; external cut-offs still do.
- [ ] Failed runs and unsuccessful tuning configurations are retained.
- [ ] Results, weights and runtime information are saved outside notebook memory.
- [ ] Every group member can explain the environment and all three algorithms.